Inspect Volume storage and its retry behavior without a remote connection.


In [ ]:
from setup.nbs.inspect_tools import show, show_methods
from pair_recon.core.backends.store import ApiVolumeStore, retry_transient, volume_failure_streak
from pair_recon_left.backends.store import VolumeStore as LeftVolumeStore
from pair_recon_right.engine.backends.store import VolumeStore as RightVolumeStore

show(retry_transient, volume_failure_streak)
show_methods(ApiVolumeStore, 'child', 'put_bytes', 'put_json', 'get_bytes', 'get_json', 'exists', 'list_keys', 'delete_keys', 'clear')
show_methods(LeftVolumeStore, 'put_bytes', 'get_bytes', 'exists')
show_methods(RightVolumeStore, 'put_bytes', 'get_bytes', 'exists')


## Toy example

Run this cell locally to inspect the concrete inputs and outputs.


In [ ]:
calls = []
def operation():
    calls.append("attempt")
    if len(calls) == 1:
        raise ConnectionError("temporary")
    return "ok"
from unittest.mock import patch
with patch("pair_recon.core.backends.store.time.sleep"):
    print("result:", retry_transient("toy", operation))
print("attempts:", len(calls))


## `ApiVolumeStore` and Left store with a fake Files API

The fake records byte payloads in memory. Each input and returned value is printed; no Databricks API is called.


In [ ]:
import io
from types import SimpleNamespace

class FakeFiles:
    def __init__(self):
        self.data = {}
    def create_directory(self, path):
        pass
    def upload(self, path, content, overwrite):
        self.data[path] = content.read()
    def download(self, path):
        if path not in self.data:
            raise FileNotFoundError(path)
        return SimpleNamespace(contents=io.BytesIO(self.data[path]))
    def get_metadata(self, path):
        if path not in self.data:
            raise FileNotFoundError(path)
        return SimpleNamespace(path=path)
    def list_directory_contents(self, path):
        return [SimpleNamespace(path=key, is_directory=False) for key in self.data if key.startswith(path.rstrip("/") + "/")]
    def delete(self, path):
        del self.data[path]

files = FakeFiles()
store = ApiVolumeStore("/Volumes/demo/recon/runs", files=files)
child = store.child("toy-run")
print("child root:", child.root)
print("put bytes:", child.put_bytes("sample.txt", b"hello"))
print("get bytes:", child.get_bytes("sample.txt"))
print("put json:", child.put_json("state.json", {"state": "ready"}))
print("get json:", child.get_json("state.json"))
print("exists:", child.exists("state.json"), child.exists("missing.json"))
print("list keys:", child.list_keys(""))
print("delete keys:", child.delete_keys(["sample.txt"]))
print("clear remaining:", child.clear())
print("remaining fake files:", files.data)
left = LeftVolumeStore("/Volumes/demo/recon/left", files=files)
print("Left put:", left.put_bytes("toy.txt", b"left"))
print("Left get:", left.get_bytes("toy.txt"), "exists:", left.exists("toy.txt"))
print("failure streak after successful calls:", volume_failure_streak())


## Right `VolumeStore` with a temporary directory

Right accesses files through a mounted directory. A temporary local directory makes the filesystem behavior visible without using a real Volume.


In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as directory:
    right = RightVolumeStore(directory)
    print("input bytes:", b"right")
    print("put:", right.put_bytes("toy.txt", b"right"))
    print("get:", right.get_bytes("toy.txt"))
    print("exists:", right.exists("toy.txt"), right.exists("missing.txt"))
